# Monthly Regional Forecast — Handovers, Returns & New Stock

Three Prophet models at **region × month** level for three flows.

## Structure
1. [Imports & data loading](#imports)
2. [Configuration](#config) ← **edit this cell to experiment**
3. [Feature engineering](#features) — active fleet trend + seasonal peaks
4. [departure_handover](#handover) — model build → CV → grid search
5. [arrival_return](#returns) — model build → CV → grid search
6. [arrival_new_stock](#newstock) — model build → CV → grid search
7. [Summary comparison](#summary)
8. [Forecast TARGET_MONTH](#forecast)

In [1]:
import sys, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from src.forecasting import MonthlyFlowForecaster
from src.config import REGIONS, DEFAULT_PARAM_GRID


In [2]:
car_events = pd.read_csv('../data/car_events_corrected.csv')
compounds  = pd.read_csv('../data/compounds.csv')
demand     = pd.read_csv('../data/demand_forecast_inputs.csv')

# Add region to events, parse dates
car_events['event_date'] = pd.to_datetime(car_events['event_date'])
if 'region' not in car_events.columns:
    car_events = car_events.merge(compounds[['compound_id', 'region']], on='compound_id', how='left')

car_events['month'] = car_events['event_date'].dt.to_period('M')

# Monthly regional actuals
actuals = car_events.groupby(['month', 'region']).agg(
    real_handovers = ('event_type', lambda x: (x == 'departure_handover').sum()),
    real_returns   = ('event_type', lambda x: (x == 'arrival_return').sum()),
    real_new_stock = ('event_type', lambda x: (x == 'arrival_new_stock').sum()),
).reset_index()

demand['month'] = pd.to_datetime(demand['month']).dt.to_period('M')
df = demand.merge(actuals, on=['month', 'region'], how='left')

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

print(f'Loaded {len(df)} rows · {df["month"].nunique()} months · {df["region"].nunique()} regions')
print(df[df['region'] == 'South'][['month', 'planned_handovers', 'real_handovers', 'real_returns', 'real_new_stock']].tail(4).to_string(index=False))

Loaded 96 rows · 24 months · 4 regions
  month  planned_handovers  real_handovers  real_returns  real_new_stock
2025-09                370            1213          1202             300
2025-10                367            1320          1255             276
2025-11                375            1190          1130             251
2025-12                294            1271          1249             282


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — edit this cell to experiment
# ─────────────────────────────────────────────────────────────────────────────

# ── Seasonal peak indicator ───────────────────────────────────────────────────
# Months that are systematically underforecast (from residual analysis).
# is_peak_month=1 for these months
PEAK_MONTHS = {1, 7, 9, 12}  

# Regions where is_peak_month IMPROVES MAPE (from grid search).
# North: -1.1pp  West: -0.3pp  South: +0.5pp worse  East: +0.8pp worse
PEAK_MONTH_REGIONS = {'North', 'West'}

# ── Regressors ────────────────────────────────────────────────────────────────
# To add active_fleet_lag1: append 'active_fleet_lag1' to the list below.
# is_peak_month is added automatically for regions in PEAK_MONTH_REGIONS.
HO_REGRESSORS  = ['new_subs_lag1', 'planned_ho_lag1']
RET_REGRESSORS = ['returns_lag1', 'planned_ret_lag0', 'new_subs_lag1', 'backlog_lag1']
NS_REGRESSORS  = ['planned_ho_lag1', 'new_subs_lag1']

# ── Best hyperparameters ──────────────────────────────────────────────────────
# From walk-forward grid search.  To retune, uncomment grid search cells below.
HO_BEST = {
    # MAPE: South=3.2%, West=8.9%, North=7.4%, East=7.0%
    # (North/West use is_peak_month → updated params after re-tuning)
    'South': dict(changepoint_prior_scale=0.10, seasonality_prior_scale=20, n_changepoints=5, seasonality_mode='additive'),
    'West':  dict(changepoint_prior_scale=0.05, seasonality_prior_scale=20, n_changepoints=3, seasonality_mode='additive'),
    'North': dict(changepoint_prior_scale=0.05, seasonality_prior_scale=1,  n_changepoints=5, seasonality_mode='additive'),
    'East':  dict(changepoint_prior_scale=0.50, seasonality_prior_scale=20, n_changepoints=5, seasonality_mode='multiplicative'),
}

RET_BEST = {
    # MAPE: South=3.4%, West=8.3%, North=7.4%, East=2.8%
    'South': dict(changepoint_prior_scale=0.10, seasonality_prior_scale=1,  n_changepoints=5, seasonality_mode='multiplicative'),
    'West':  dict(changepoint_prior_scale=0.05, seasonality_prior_scale=1,  n_changepoints=3, seasonality_mode='multiplicative'),
    'North': dict(changepoint_prior_scale=0.05, seasonality_prior_scale=1,  n_changepoints=5, seasonality_mode='additive'),
    'East':  dict(changepoint_prior_scale=0.30, seasonality_prior_scale=1,  n_changepoints=5, seasonality_mode='additive'),
}

NS_BEST = {
    # MAPE: South=6.0%, West=6.9%, North=10.0%, East=3.6%
    'South': dict(changepoint_prior_scale=0.05, seasonality_prior_scale=1,  n_changepoints=8, seasonality_mode='additive'),
    'West':  dict(changepoint_prior_scale=0.10, seasonality_prior_scale=10, n_changepoints=5, seasonality_mode='additive'),
    'North': dict(changepoint_prior_scale=0.05, seasonality_prior_scale=1,  n_changepoints=3, seasonality_mode='additive'),
    'East':  dict(changepoint_prior_scale=0.05, seasonality_prior_scale=1,  n_changepoints=5, seasonality_mode='additive'),
}

# ── Grid search space ─────────────────────────────────────────────────────────
# Modify to focus on a smaller space for faster re-tuning.
PARAM_GRID = DEFAULT_PARAM_GRID.copy()

In [ ]:
_ARRIVALS_FLEET   = {'arrival_new_stock', 'arrival_return'}
_DEPARTURES_FLEET = {'departure_handover', 'departure_remarketing'}
_FLEET_TYPES      = _ARRIVALS_FLEET | _DEPARTURES_FLEET

fleet_ev = car_events[car_events['event_type'].isin(_FLEET_TYPES)].copy()
fleet_ev['signed'] = fleet_ev['event_type'].map(
    lambda t: +1 if t in _ARRIVALS_FLEET else -1
)
fleet_monthly = (
    fleet_ev.groupby(['month', 'region'])['signed'].sum()
    .reset_index(name='net_flow')
    .sort_values(['region', 'month'])
)
fleet_monthly['active_fleet'] = fleet_monthly.groupby('region')['net_flow'].cumsum()

df = df.merge(fleet_monthly[['month', 'region', 'active_fleet']], on=['month', 'region'], how='left')
df['active_fleet_lag1'] = df.groupby('region')['active_fleet'].shift(1)
df['is_peak_month'] = df['month'].dt.month.isin(PEAK_MONTHS).astype(float)

New features:
  month  active_fleet  active_fleet_lag1  is_peak_month
2025-07          1006              904.0            1.0
2025-08          1032             1006.0            0.0
2025-09          1147             1032.0            1.0
2025-10          1173             1147.0            0.0
2025-11          1199             1173.0            0.0
2025-12          1254             1199.0            1.0

active_fleet range: -955 – 1254


---
## 1. departure_handover

**Regressors**: `new_subs_lag1`, `planned_ho_lag1` (+ `is_peak_month` for North/West)

Key insight: subscription signals lead by 1 month; more regressors overfit on ~24 data points.  
`is_peak_month` captures systematic underprediction in Jan/Jul/Sep/Dec (−1.1pp North, −0.3pp West).


In [ ]:
def build_ho_model_df(region: str) -> pd.DataFrame:
    """Build Prophet-ready DataFrame for departure_handover in `region`."""
    regressors = HO_REGRESSORS + (['is_peak_month'] if region in PEAK_MONTH_REGIONS else [])
    sub = df[df['region'] == region].sort_values('month').copy()
    sub['ds']              = sub['month'].dt.to_timestamp()
    sub['new_subs_lag1']   = sub['new_subscriptions'].shift(1)
    sub['planned_ho_lag1'] = sub['planned_handovers'].shift(1)
    sub.rename(columns={'real_handovers': 'y'}, inplace=True)
    return sub[['ds', 'y'] + regressors].dropna().reset_index(drop=True)

ho_model_dfs = {region: build_ho_model_df(region) for region in REGIONS}

for region, mdf in ho_model_dfs.items():
    regs = HO_REGRESSORS + (['is_peak_month'] if region in PEAK_MONTH_REGIONS else [])
    print(f'{region:8s}  rows={len(mdf)}  regressors={regs}')
    print(mdf[['ds', 'y'] + regs[:2]].tail(3).to_string(index=False))
    print()

In [6]:
handover_results = {}

for region in REGIONS:
    mdf        = ho_model_dfs[region]
    regressors = HO_REGRESSORS + (['is_peak_month'] if region in PEAK_MONTH_REGIONS else [])
    changepoints = MANUAL_CHANGEPOINTS['departure_handover'].get(region)

    cv  = MonthlyFlowForecaster(region, 'departure_handover', mdf, HO_BEST[region],
                                regressors, changepoints=changepoints).run_cv()
    mae = cv['abs_error'].mean()
    mape = cv['ape_pct'].mean()
    cov  = ((cv['actual'] >= cv['lower_95']) & (cv['actual'] <= cv['upper_95'])).mean() * 100
    handover_results[region] = {'cv': cv, 'model_df': mdf}

    print(f'{region:8s}  MAPE={mape:.1f}%  MAE={mae:.0f}  95%CI={cov:.0f}%')
    print(cv[['date', 'actual', 'predicted', 'error', 'ape_pct']].to_string(index=False))
    print()


01:02:46 - cmdstanpy - INFO - Chain [1] start processing


01:02:47 - cmdstanpy - INFO - Chain [1] done processing


01:02:47 - cmdstanpy - INFO - Chain [1] start processing


01:02:47 - cmdstanpy - INFO - Chain [1] done processing


01:02:47 - cmdstanpy - INFO - Chain [1] start processing


...

01:02:47 - cmdstanpy - INFO - Chain [1] done processing


01:02:47 - cmdstanpy - INFO - Chain [1] start processing


North     MAPE=7.3%  MAE=41  95%CI=80%
   date  actual  predicted  error  ape_pct
2025-08     567      554.0   13.0      2.3
2025-09     599      557.3   41.7      7.0
2025-10     593      550.6   42.4      7.2
2025-11     509      585.0  -76.0     14.9
2025-12     616      585.1   30.9      5.0



01:02:47 - cmdstanpy - INFO - Chain [1] done processing


East      MAPE=7.9%  MAE=57  95%CI=80%
   date  actual  predicted  error  ape_pct
2025-08     765      762.0    3.0      0.4
2025-09     778      762.1   15.9      2.0
2025-10     835      739.6   95.4     11.4
2025-11     658      788.1 -130.1     19.8
2025-12     708      748.8  -40.8      5.8



In [7]:
# ── Handover grid search — uncomment to retune, ~10 min ──────────────────────
# Run this after changing HO_REGRESSORS or PEAK_MONTH_REGIONS.
# Copy best params back to HO_BEST in the config cell.

# HO_GRID = {}
# for region in REGIONS:
#     regressors = HO_REGRESSORS + (['is_peak_month'] if region in PEAK_MONTH_REGIONS else [])
#     changepoints = MANUAL_CHANGEPOINTS['departure_handover'].get(region)
#     print(f'\n── {region} (regressors={regressors})')
#     grid = MonthlyFlowForecaster.run_grid_search(
#         ho_model_dfs[region], regressors,
#         param_grid=PARAM_GRID, loss='mape',
#         extra_changepoints=changepoints,
#     )
#     HO_GRID[region] = grid
#
# print('\nBest params — paste into HO_BEST in config cell:')
# for region in REGIONS:
#     best = HO_GRID[region].iloc[0]
#     print(f"    '{region}': dict(changepoint_prior_scale={best['changepoint_prior_scale']}, "
#           f"seasonality_prior_scale={best['seasonality_prior_scale']}, "
#           f"n_changepoints={int(best['n_changepoints'])}, "
#           f"seasonality_mode='{best['seasonality_mode']}'),")

print('Grid search ready — uncomment to run.')

Grid search ready — uncomment to run.


---
## 2. arrival_return

**Regressors**: `returns_lag1`, `planned_ret_lag0`, `new_subs_lag1`, `backlog_lag1`



In [ ]:
def build_ret_model_df(region: str) -> pd.DataFrame:
    """Build Prophet-ready DataFrame for arrival_return in `region`."""
    sub = df[df['region'] == region].sort_values('month').copy()
    sub['ds']               = sub['month'].dt.to_timestamp()
    sub['returns_lag1']     = sub['real_returns'].shift(1)
    sub['planned_ret_lag0'] = sub['planned_returns']
    sub['new_subs_lag1']    = sub['new_subscriptions'].shift(1)
    sub['backlog_lag1']     = sub['backlog_orders'].shift(1)
    sub.rename(columns={'real_returns': 'y'}, inplace=True)
    return sub[['ds', 'y'] + RET_REGRESSORS].dropna().reset_index(drop=True)

ret_model_dfs = {region: build_ret_model_df(region) for region in REGIONS}

for region, mdf in ret_model_dfs.items():
    print(f'{region:8s}  rows={len(mdf)}  regressors={RET_REGRESSORS}')

In [9]:
returns_results = {}

for region in REGIONS:
    mdf          = ret_model_dfs[region]
    changepoints = MANUAL_CHANGEPOINTS['arrival_return'].get(region)

    cv   = MonthlyFlowForecaster(region, 'arrival_return', mdf, RET_BEST[region],
                                 RET_REGRESSORS, changepoints=changepoints).run_cv()
    mae  = cv['abs_error'].mean()
    mape = cv['ape_pct'].mean()
    cov  = ((cv['actual'] >= cv['lower_95']) & (cv['actual'] <= cv['upper_95'])).mean() * 100
    returns_results[region] = {'cv': cv, 'model_df': mdf}

    print(f'{region:8s}  MAPE={mape:.1f}%  MAE={mae:.0f}  95%CI={cov:.0f}%')
    print(cv[['date', 'actual', 'predicted', 'error', 'ape_pct']].to_string(index=False))
    print()


01:02:48 - cmdstanpy - INFO - Chain [1] start processing


01:02:48 - cmdstanpy - INFO - Chain [1] done processing


01:02:48 - cmdstanpy - INFO - Chain [1] start processing


01:02:48 - cmdstanpy - INFO - Chain [1] done processing


01:02:48 - cmdstanpy - INFO - Chain [1] start processing


...

01:02:48 - cmdstanpy - INFO - Chain [1] done processing


North     MAPE=7.4%  MAE=41  95%CI=60%
   date  actual  predicted  error  ape_pct
2025-08     582      595.7  -13.7      2.4
2025-09     563      594.3  -31.3      5.6
2025-10     557      578.0  -21.0      3.8
2025-11     505      572.2  -67.2     13.3
2025-12     610      537.9   72.1     11.8



01:02:48 - cmdstanpy - INFO - Chain [1] start processing


01:02:48 - cmdstanpy - INFO - Chain [1] done processing


East      MAPE=2.8%  MAE=18  95%CI=80%
   date  actual  predicted  error  ape_pct
2025-08     666      673.3   -7.3      1.1
2025-09     665      679.4  -14.4      2.2
2025-10     691      660.7   30.3      4.4
2025-11     625      660.3  -35.3      5.6
2025-12     697      701.0   -4.0      0.6



In [10]:
# ── Returns grid search — uncomment to retune, ~10 min ───────────────────────
# Copy best params back to RET_BEST in the config cell.

# RET_GRID = {}
# for region in REGIONS:
#     changepoints = MANUAL_CHANGEPOINTS['arrival_return'].get(region)
#     print(f'\n── {region}')
#     grid = MonthlyFlowForecaster.run_grid_search(
#         ret_model_dfs[region], RET_REGRESSORS,
#         param_grid=PARAM_GRID, loss='mape',
#         extra_changepoints=changepoints,
#     )
#     RET_GRID[region] = grid
#
# print('\nBest params — paste into RET_BEST in config cell:')
# for region in REGIONS:
#     best = RET_GRID[region].iloc[0]
#     print(f"    '{region}': dict(changepoint_prior_scale={best['changepoint_prior_scale']}, "
#           f"seasonality_prior_scale={best['seasonality_prior_scale']}, "
#           f"n_changepoints={int(best['n_changepoints'])}, "
#           f"seasonality_mode='{best['seasonality_mode']}'),")

print('Grid search ready — uncomment to run.')

Grid search ready — uncomment to run.


---
## 3. arrival_new_stock

**Regressors**: `planned_ho_lag1`, `new_subs_lag1`

Very smooth flow (CoV=0.08). Cars are procured ahead of planned handovers, so
`planned_handovers` at lag1–2 is the strongest signal.


In [ ]:
def build_ns_model_df(region: str) -> pd.DataFrame:
    """Build Prophet-ready DataFrame for arrival_new_stock in `region`."""
    sub = df[df['region'] == region].sort_values('month').copy()
    sub['ds']              = sub['month'].dt.to_timestamp()
    sub['planned_ho_lag1'] = sub['planned_handovers'].shift(1)
    sub['new_subs_lag1']   = sub['new_subscriptions'].shift(1)
    sub.rename(columns={'real_new_stock': 'y'}, inplace=True)
    return sub[['ds', 'y'] + NS_REGRESSORS].dropna().reset_index(drop=True)

ns_model_dfs = {region: build_ns_model_df(region) for region in REGIONS}

for region, mdf in ns_model_dfs.items():
    print(f'{region:8s}  rows={len(mdf)}  regressors={NS_REGRESSORS}')

In [12]:
new_stock_results = {}

for region in REGIONS:
    mdf          = ns_model_dfs[region]
    changepoints = MANUAL_CHANGEPOINTS['arrival_new_stock'].get(region)

    cv   = MonthlyFlowForecaster(region, 'arrival_new_stock', mdf, NS_BEST[region],
                                 NS_REGRESSORS, changepoints=changepoints).run_cv()
    mae  = cv['abs_error'].mean()
    mape = cv['ape_pct'].mean()
    cov  = ((cv['actual'] >= cv['lower_95']) & (cv['actual'] <= cv['upper_95'])).mean() * 100
    new_stock_results[region] = {'cv': cv, 'model_df': mdf}

    print(f'{region:8s}  MAPE={mape:.1f}%  MAE={mae:.0f}  95%CI={cov:.0f}%')
    print(cv[['date', 'actual', 'predicted', 'error', 'ape_pct']].to_string(index=False))
    print()


01:02:49 - cmdstanpy - INFO - Chain [1] start processing


01:02:49 - cmdstanpy - INFO - Chain [1] done processing


01:02:49 - cmdstanpy - INFO - Chain [1] start processing


01:02:49 - cmdstanpy - INFO - Chain [1] done processing


01:02:49 - cmdstanpy - INFO - Chain [1] start processing


...

01:02:49 - cmdstanpy - INFO - Chain [1] done processing


01:02:49 - cmdstanpy - INFO - Chain [1] start processing


01:02:49 - cmdstanpy - INFO - Chain [1] done processing


North     MAPE=10.0%  MAE=13  95%CI=40%
   date  actual  predicted  error  ape_pct
2025-08     115      130.5  -15.5     13.4
2025-09     126      123.9    2.1      1.6
2025-10     144      118.7   25.3     17.5
2025-11     113      129.2  -16.2     14.3
2025-12     130      125.9    4.1      3.1



East      MAPE=3.6%  MAE=5  95%CI=100%
   date  actual  predicted  error  ape_pct
2025-08     147      146.2    0.8      0.5
2025-09     138      150.9  -12.9      9.3
2025-10     148      148.2   -0.2      0.1
2025-11     140      151.0  -11.0      7.9
2025-12     152      152.3   -0.3      0.2



In [13]:
# ── New stock grid search — uncomment to retune, ~10 min ─────────────────────
# Copy best params back to NS_BEST in the config cell.

# NS_GRID = {}
# for region in REGIONS:
#     changepoints = MANUAL_CHANGEPOINTS['arrival_new_stock'].get(region)
#     print(f'\n── {region}')
#     grid = MonthlyFlowForecaster.run_grid_search(
#         ns_model_dfs[region], NS_REGRESSORS,
#         param_grid=PARAM_GRID, loss='mape',
#         extra_changepoints=changepoints,
#     )
#     NS_GRID[region] = grid
#
# print('\nBest params — paste into NS_BEST in config cell:')
# for region in REGIONS:
#     best = NS_GRID[region].iloc[0]
#     print(f"    '{region}': dict(changepoint_prior_scale={best['changepoint_prior_scale']}, "
#           f"seasonality_prior_scale={best['seasonality_prior_scale']}, "
#           f"n_changepoints={int(best['n_changepoints'])}, "
#           f"seasonality_mode='{best['seasonality_mode']}'),")

print('Grid search ready — uncomment to run.')

Grid search ready — uncomment to run.


---
## 4. Summary — CV performance across all flows

In [ ]:

print('Walk-forward CV MAPE summary')
print(f'{"Region":8s}  {"Handovers":>12s}  {"Returns":>10s}  {"New Stock":>10s}')
print('─' * 48)
for region in REGIONS:
    ho_mape  = handover_results[region]['cv']['ape_pct'].mean()
    ret_mape = returns_results[region]['cv']['ape_pct'].mean()
    ns_mape  = new_stock_results[region]['cv']['ape_pct'].mean()
    print(f'{region:8s}  {ho_mape:>10.1f}%  {ret_mape:>8.1f}%  {ns_mape:>8.1f}%')

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('Walk-forward CV — Actual vs Predicted', fontsize=14, fontweight='bold')

flow_configs = [
    ('departure_handover', handover_results,   '#1f77b4'),
    ('arrival_return',     returns_results,    '#2ca02c'),
    ('arrival_new_stock',  new_stock_results,  '#ff7f0e'),
]

for row_idx, (flow_label, results, color) in enumerate(flow_configs):
    for col_idx, region in enumerate(REGIONS):
        ax  = axes[row_idx, col_idx]
        cv  = results[region]['cv']
        ax.plot(cv['date'], cv['actual'],    'k-o', ms=4, label='Actual')
        ax.plot(cv['date'], cv['predicted'], color=color, marker='s', ms=4, ls='--', label='Predicted')
        mape = cv['ape_pct'].mean()
        ax.set_title(f'{region} — {flow_label.split("_")[1]}\nMAPE={mape:.1f}%', fontsize=9)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        if col_idx == 0:
            ax.legend(fontsize=7)

plt.tight_layout()
plt.show()